In [1]:
from pathlib import Path

ROOT = Path("/kaggle/input")

print("=" * 70)
print("KAGGLE INPUT DATASETS")
print("=" * 70)

for item in sorted(ROOT.iterdir()):
    if item.is_dir():
        files = [p for p in item.rglob("*") if p.is_file()]
        dirs = [p for p in item.rglob("*") if p.is_dir()]

        print(f"\nDataset/folder: {item.name}")
        print(f"Path: {item}")
        print(f"Files: {len(files):,}")
        print(f"Subfolders: {len(dirs):,}")

        print("Sample files:")
        for p in files[:8]:
            print(f"  {p.relative_to(item)}")

print("\nInspection complete.")

KAGGLE INPUT DATASETS

Dataset/folder: datasets
Path: /kaggle/input/datasets
Files: 6,233
Subfolders: 3
Sample files:
  pranavsunilwagh/nd-iris/NDCLD15/06312d1143.tiff
  pranavsunilwagh/nd-iris/NDCLD15/06391d547.tiff
  pranavsunilwagh/nd-iris/NDCLD15/06751d359.tiff
  pranavsunilwagh/nd-iris/NDCLD15/06589d622.tiff
  pranavsunilwagh/nd-iris/NDCLD15/07009d14.tiff
  pranavsunilwagh/nd-iris/NDCLD15/07030d64.tiff
  pranavsunilwagh/nd-iris/NDCLD15/07014d3465.tiff
  pranavsunilwagh/nd-iris/NDCLD15/06638d160.tiff

Inspection complete.


In [2]:

from pathlib import Path
from collections import Counter
from PIL import Image, UnidentifiedImageError
import pandas as pd
import numpy as np
import json, time, zipfile

# ============================================================
# NOTEBOOK 01 — ND IRIS DATASET COLLECTION & ANALYSIS
# ============================================================
START = time.time()
DATASET_ROOT = Path("/kaggle/input/datasets/pranavsunilwagh/nd-iris/NDCLD15")
PROJECT = Path("/kaggle/working/nd_iris_project")
OUT = PROJECT / "notebook01_dataset_analysis"
OUT.mkdir(parents=True, exist_ok=True)

if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Dataset root not found: {DATASET_ROOT}")

# 1. Inventory all files
all_files = sorted(p for p in DATASET_ROOT.rglob("*") if p.is_file())
print(f"Dataset root: {DATASET_ROOT}")
print(f"Total files: {len(all_files):,}")

extension_counts = Counter(p.suffix.lower() or "[no extension]" for p in all_files)
print("\nFile extensions:")
for ext, count in extension_counts.most_common():
    print(f"  {ext}: {count:,}")

# 2. Inspect image files and record metadata
records, errors = [], []
for i, path in enumerate(all_files, 1):
    row = {
        "filename": path.name,
        "relative_path": str(path.relative_to(DATASET_ROOT)),
        "extension": path.suffix.lower(),
        "parent_folder": path.parent.name,
        "size_bytes": path.stat().st_size,
        "width": None,
        "height": None,
        "mode": None,
        "format": None,
        "readable": False,
    }
    try:
        with Image.open(path) as im:
            row.update({
                "width": im.width,
                "height": im.height,
                "mode": im.mode,
                "format": im.format,
                "readable": True,
            })
            im.verify()
    except Exception as e:
        errors.append({
            "relative_path": row["relative_path"],
            "error": str(e)[:300]
        })
    records.append(row)

    if i % 500 == 0 or i == len(all_files):
        print(f"Inspected {i:,}/{len(all_files):,} files")

df = pd.DataFrame(records)
df.to_csv(OUT / "ND_Iris_file_metadata.csv", index=False)
pd.DataFrame(errors, columns=["relative_path", "error"]).to_csv(
    OUT / "ND_Iris_unreadable_files.csv", index=False
)

# 3. Summaries
readable = df[df["readable"]]
dimension_counts = (
    readable.groupby(["width", "height"]).size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
dimension_counts.to_csv(OUT / "ND_Iris_image_dimensions.csv", index=False)

folder_counts = (
    df.groupby("parent_folder").size()
    .reset_index(name="file_count")
    .sort_values("file_count", ascending=False)
)
folder_counts.to_csv(OUT / "ND_Iris_folder_counts.csv", index=False)

# Filename-derived IDs are only exploratory; do not assume these are true identities.
# We record filename patterns for later verification instead of inventing labels.
filename_examples = df["filename"].head(30).tolist()
tiff_count = int(df["extension"].isin([".tif", ".tiff"]).sum())

summary = {
    "dataset_name": "ND Iris / NDCLD15",
    "dataset_root": str(DATASET_ROOT),
    "total_files": int(len(df)),
    "extension_counts": {str(k): int(v) for k, v in extension_counts.items()},
    "readable_images": int(df["readable"].sum()),
    "unreadable_or_unsupported_files": int(len(errors)),
    "tiff_files": tiff_count,
    "dimension_counts": [
        {"width": int(r.width), "height": int(r.height), "count": int(r["count"])}
        for _, r in dimension_counts.iterrows()
    ],
    "parent_folder_counts": [
        {"folder": str(r.parent_folder), "count": int(r.file_count)}
        for _, r in folder_counts.iterrows()
    ],
    "sample_filenames": filename_examples,
    "identity_labels_verified": False,
    "notes": [
        "Filename patterns and folder names have not yet been validated as identity labels.",
        "No train/validation/test split has been created in this notebook.",
        "Dataset licensing and permitted research use should be checked before redistribution."
    ],
}
with open(OUT / "ND_Iris_dataset_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# 4. Conclusion cell output, written into a report
width_height = (
    f"{int(readable['width'].nunique())} distinct widths and "
    f"{int(readable['height'].nunique())} distinct heights"
    if len(readable) else "No readable images"
)
conclusion = f"""# Notebook 01 — ND Iris Dataset Analysis: Conclusion

- Dataset root: `{DATASET_ROOT}`
- Total files discovered: **{len(df):,}**
- TIFF files: **{tiff_count:,}**
- Readable images: **{int(df['readable'].sum()):,}**
- Unreadable or unsupported files: **{len(errors):,}**
- Image dimensions: {width_height}

## Interpretation
This notebook inventories the dataset and records image metadata. Identity labels have
not been assumed from filenames or folder names. Before preprocessing and recognition,
the dataset's subject/eye identity conventions and any left/right-eye structure must be
verified. No model performance or privacy claims can be made from this audit alone.

## Outputs
- `ND_Iris_file_metadata.csv`
- `ND_Iris_unreadable_files.csv`
- `ND_Iris_image_dimensions.csv`
- `ND_Iris_folder_counts.csv`
- `ND_Iris_dataset_summary.json`
"""
(OUT / "Notebook01_Conclusion.md").write_text(conclusion)

# 5. ZIP results and verify integrity
zip_path = PROJECT / "ND_Iris_Notebook01_Results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            z.write(p, arcname=f"notebook01_dataset_analysis/{p.relative_to(OUT)}")

with zipfile.ZipFile(zip_path, "r") as z:
    bad_member = z.testzip()
    archived = z.namelist()

print("\n" + "=" * 65)
print("NOTEBOOK 01 RESULTS")
print("=" * 65)
print(f"Files discovered: {len(df):,}")
print(f"Readable images: {int(df['readable'].sum()):,}")
print(f"Unreadable/unsupported: {len(errors):,}")
print(f"Most common dimensions:\n{dimension_counts.head(10).to_string(index=False)}")
print(f"\nConclusion: {OUT / 'Notebook01_Conclusion.md'}")
print(f"ZIP: {zip_path}")
print(f"Archived files: {len(archived)}")
print(f"ZIP integrity: {'PASSED' if bad_member is None else 'FAILED: ' + str(bad_member)}")
print(f"Elapsed: {(time.time()-START)/60:.2f} minutes")
print("\nNOTE: Filename-derived identity labels have NOT been validated.")


Dataset root: /kaggle/input/datasets/pranavsunilwagh/nd-iris/NDCLD15
Total files: 6,233

File extensions:
  .tiff: 6,233
Inspected 500/6,233 files
Inspected 1,000/6,233 files
Inspected 1,500/6,233 files
Inspected 2,000/6,233 files
Inspected 2,500/6,233 files
Inspected 3,000/6,233 files
Inspected 3,500/6,233 files
Inspected 4,000/6,233 files
Inspected 4,500/6,233 files
Inspected 5,000/6,233 files
Inspected 5,500/6,233 files
Inspected 6,000/6,233 files
Inspected 6,233/6,233 files

NOTEBOOK 01 RESULTS
Files discovered: 6,233
Readable images: 6,233
Unreadable/unsupported: 0
Most common dimensions:
 width  height  count
   640     480   6233

Conclusion: /kaggle/working/nd_iris_project/notebook01_dataset_analysis/Notebook01_Conclusion.md
ZIP: /kaggle/working/nd_iris_project/ND_Iris_Notebook01_Results.zip
Archived files: 6
ZIP integrity: PASSED
Elapsed: 0.93 minutes

NOTE: Filename-derived identity labels have NOT been validated.
